# KOSIS_API_v3 - KOSIS 통계 수집 → investar 적재

KOSIS(국가통계포털) 통계표를 받아 investar에 저장합니다. 소매판매액지수로 시작하고, 다른 KOSIS 통계표도 **등록표에 한 줄 추가**하면 같은 테이블에 쌓입니다.

**저장 테이블 (investar, 표준 규격 STD-1)**

| 테이블 | 내용 | 키 |
|---|---|---|
| `kr_kosis_series_info` | 시리즈 설명표 (통계표, 항목, 분류 C1~C8, 단위, 통계표 갱신일) | series_id |
| `kr_kosis_data` | 값 최신본 (시리즈 x 기간 1행) | series_id, period_end |
| `kr_kosis_data_vintage` | 값이 처음 들어오거나 바뀐 이력 | id |

- `series_id` = `kosis_` + (기관·통계표·항목·분류코드) 해시 16자리. 같은 항목·분류는 항상 같은 series_id
- STD-1: 소문자 snake_case / utf8mb4_general_ci / `period_end`(기간 말일) + `freq` / 수집 시각 KST / 이력 `_vintage` / 단위는 설명표

**v3**: KEYS.py 경로를 직접 지정해서 불러옴 (1번 셀 `KEYS_PATH`)

**v1에서 바뀐 점**
- v1 첫 셀의 `statisticsData.do`는 '사용자 등록 통계'용 주소라 `필수요청변수 누락(20)` 오류가 났습니다 → 통계표 직접 조회 주소(`Param/statisticsParameterData.do`)로 통일
- 분류 단계(objL1~objL8)와 주기(M/Q/Y)를 자동으로 찾음. 4만 셀 제한에 걸리면 기간을 자동으로 쪼개 받음
- API 키를 코드에서 빼고 `DATA/KEYS.py`에서 읽음. 오류 메시지의 키는 `***`로 가림
- 처음엔 전체 기간, 이후엔 DB 마지막 시점 기준 최근 몇 기간만 갱신

**사용 순서**: 1~4번 셀 → 5번 통계표 확인(이름이 맞는지 보기) → 6번 수집·적재 → 7번 검증 → 8번 분석

## 1. 설정 - 수집할 통계표 등록표

In [1]:
# -*- coding: utf-8 -*-
import os
import sys
from pathlib import Path

# ============================================================
# 수집 대상 KOSIS 통계표 (한 줄 = 통계표 하나)
# ============================================================
#   alias   : 분석용 짧은 이름 (DB의 table_alias 로 저장)
#   org_id  : 기관 ID,  tbl_id : 통계표 ID   (KOSIS 통계표 화면 > OpenAPI 버튼에서 확인)
#   prd_se  : 주기 "M"(월) "Q"(분기) "Y"(연). None 이면 자동 탐지
#   itm_id  : 항목. "ALL" 이면 전체
#   obj     : 분류 단계. None 이면 자동 탐지 (모든 단계 ALL)
#   start   : 처음 수집 시작 시점 (월 YYYYMM, 분기 YYYYQQ, 연 YYYY)
#   enabled : False 면 건너뜀
KOSIS_TABLES = [
    {"alias": "retail_sales_index", "org_id": "101", "tbl_id": "DT_1K41012",
     "prd_se": "M", "itm_id": "ALL", "obj": None, "start": "201001", "enabled": True},
    # v1 노트북에서 조회하던 통계표. 5번 셀에서 통계표 이름을 확인한 뒤 필요 없으면 False 로
    {"alias": "kosis_376_sdmp042v", "org_id": "376", "tbl_id": "DT_376_90_SDMP042V_1",
     "prd_se": None, "itm_id": "ALL", "obj": None, "start": "201001", "enabled": True},
]

# 수집 기간
MODE = "auto"          # "auto": DB에 있으면 마지막 시점 - REFRESH_PERIODS 부터, 없으면 start 부터 / "full": 항상 start 부터
REFRESH_PERIODS = 3    # 수정치 반영용으로 다시 받을 기간 수 (월 3개월, 분기 3분기 ...)

# KOSIS API 키: 아래 경로의 KEYS.py 를 직접 불러와 KEYS[KEY_NAME] 값을 사용 (없으면 환경변수 KOSIS_API_KEY)
#   DB 접속용 config.py 도 같은 DATA 폴더에서 불러옴
KEYS_PATH = r"/DATA/KEYS.py"
KEY_NAME = "KOSIS"

SAVE_DB = True
# CSV 저장: 당분간 사용 안 함 → 기본 False. 켜면 OUT_DIR 에 통계표별 long CSV 저장
SAVE_CSV = False
OUT_DIR = r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\miscell"

TIMEOUT = 60
SLEEP_SEC = 0.3

print("등록 통계표:", sum(t["enabled"] for t in KOSIS_TABLES), "개 사용")

등록 통계표: 2 개 사용


## 2. 접속 준비 (DATA 폴더, API 키, DB)

In [2]:
import importlib.util


def load_keys_from_path(path):
    """KEYS.py 를 파일 경로로 직접 불러와 KEYS 딕셔너리를 반환."""
    spec_ = importlib.util.spec_from_file_location("investar_keys", path)
    mod = importlib.util.module_from_spec(spec_)
    spec_.loader.exec_module(mod)
    return getattr(mod, "KEYS", {})


def setup_universal_paths():
    """KEYS_PATH 의 DATA 폴더를 우선 사용, 없으면 현재 폴더에서 상위로 DATA 폴더를 찾는다."""
    cand = Path(KEYS_PATH).parent if KEYS_PATH else None
    if not (cand and cand.exists()):
        cand = next((p / "DATA" for p in [Path.cwd(), *Path.cwd().parents] if (p / "DATA").exists()), None)
    if cand is None:
        return None
    for p in (str(cand.parent), str(cand)):
        if p not in sys.path:
            sys.path.insert(0, p)
    return cand


DATA_DIR = setup_universal_paths()
print("DATA 폴더:", DATA_DIR)

keys_file = Path(KEYS_PATH) if KEYS_PATH and Path(KEYS_PATH).exists() else (DATA_DIR / "KEYS.py" if DATA_DIR else None)
KEYS = load_keys_from_path(keys_file) if keys_file and keys_file.exists() else {}
KOSIS_API_KEY = KEYS.get(KEY_NAME) or os.environ.get("KOSIS_API_KEY")
if not KOSIS_API_KEY:
    raise RuntimeError(
        f"KOSIS API 키가 없습니다. KEYS.py({keys_file}) 에 KEYS['{KEY_NAME}'] 가 없습니다.\n"
        f"  KEYS.py 에 있는 키 이름: {sorted(KEYS.keys())}\n"
        f"  위 이름 중 KOSIS 키가 있으면 1번 셀의 KEY_NAME 을 그 이름으로 바꾸세요.")
print(f"API 키 로드 완료 (KEYS['{KEY_NAME}'] from {keys_file})")

engine = None
if SAVE_DB:
    import config
    engine = config.get_engine(config.get_db_info())
    print("DB 엔진 준비 완료")

DATA 폴더: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


RuntimeError: KOSIS API 키가 없습니다. KEYS.py(C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\KEYS.py) 에 KEYS['KOSIS'] 가 없습니다.
  KEYS.py 에 있는 키 이름: []
  위 이름 중 KOSIS 키가 있으면 1번 셀의 KEY_NAME 을 그 이름으로 바꾸세요.

## 3. KOSIS API 함수

In [ ]:
import json
import time
import hashlib
from datetime import date, datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests

KOSIS_PARAM_URL = "https://kosis.kr/openapi/Param/statisticsParameterData.do"
KST = timezone(timedelta(hours=9))


class KosisError(Exception):
    def __init__(self, code, msg):
        super().__init__(f"KOSIS 오류 {code}: {msg}")
        self.code = str(code)


def _mask(text_):
    s = str(text_)
    if KOSIS_API_KEY:
        from urllib.parse import quote
        for k in {KOSIS_API_KEY, quote(KOSIS_API_KEY, safe="")}:
            s = s.replace(k, "***")
    return s


def kosis_get(org_id, tbl_id, prd_se, itm_id="ALL", obj=None, start=None, end=None, latest_n=None):
    """통계표 직접 조회. 결과 없음(30)은 빈 리스트, 그 외 오류는 KosisError."""
    params = {"method": "getList", "apiKey": KOSIS_API_KEY, "format": "json", "jsonVD": "Y",
              "orgId": org_id, "tblId": tbl_id, "prdSe": prd_se, "itmId": itm_id}
    for lv in range(1, 9):
        params[f"objL{lv}"] = (obj or {}).get(f"objL{lv}", "")
    if latest_n:
        params["newEstPrdCnt"] = latest_n
    else:
        if start:
            params["startPrdDe"] = start
        if end:
            params["endPrdDe"] = end
    try:
        r = requests.get(KOSIS_PARAM_URL, params=params, timeout=TIMEOUT)
    except requests.RequestException as e:
        raise KosisError("NET", _mask(e))
    if r.status_code != 200:
        raise KosisError(f"HTTP{r.status_code}", r.reason)
    try:
        data = r.json()
    except ValueError:
        raise KosisError("PARSE", _mask(r.text[:200]))
    if isinstance(data, dict) and "err" in data:
        code = str(data.get("err"))
        if code == "30":                      # 조회 결과 없음
            return []
        raise KosisError(code, data.get("errMsg"))
    if not isinstance(data, list):
        raise KosisError("TYPE", _mask(str(data)[:200]))
    return data


def is_too_many(e):
    """4만 셀 제한 초과 여부."""
    return isinstance(e, KosisError) and (e.code == "31" or "40,000" in str(e) or "초과" in str(e))


def detect_structure(spec):
    """주기(prd_se)와 분류 단계(objL1..k = ALL)를 최근 1개 시점 조회로 찾는다."""
    prds = [spec["prd_se"]] if spec.get("prd_se") else ["M", "Q", "Y"]
    last_err = None
    for prd in prds:
        if spec.get("obj"):
            candidates = [spec["obj"]]
        else:
            candidates = [{f"objL{lv}": "ALL" for lv in range(1, k + 1)} for k in range(1, 9)]
        for obj in candidates:
            try:
                rows = kosis_get(spec["org_id"], spec["tbl_id"], prd, spec.get("itm_id", "ALL"), obj, latest_n=1)
            except KosisError as e:
                last_err = e
                if e.code in ("20", "21"):     # 분류 단계 부족 → 다음 단계 포함해 재시도
                    continue
                raise
            if rows:
                return prd, obj, rows
            time.sleep(SLEEP_SEC)
    raise KosisError("DETECT", f"주기/분류 탐지 실패 ({spec['tbl_id']}). 마지막 오류: {last_err}")


def _shift_period(p, prd, n):
    """기간 문자열 p 를 n 기간 이동."""
    if prd == "M":
        return (pd.Period(f"{p[:4]}-{p[4:6]}", "M") + n).strftime("%Y%m")
    if prd == "Q":
        q = pd.Period(f"{p[:4]}Q{int(p[4:])}", "Q") + n
        return f"{q.year}{q.quarter:02d}"
    if prd == "Y":
        return str(int(p[:4]) + n)
    return p


def _period_list(start, end, prd):
    out, cur = [], start
    while cur <= end:
        out.append(cur)
        cur = _shift_period(cur, prd, 1)
    return out


def fetch_range(spec, prd, obj, start, end, chunk=None):
    """start~end 를 나눠 받는다. 4만 셀 초과면 구간을 절반으로 쪼개 재시도."""
    periods = _period_list(start, end, prd)
    chunk = chunk or {"M": 24, "Q": 20, "Y": 20}.get(prd, 12)
    rows = []
    stack = [periods[i:i + chunk] for i in range(0, len(periods), chunk)][::-1]
    while stack:
        part = stack.pop()
        try:
            got = kosis_get(spec["org_id"], spec["tbl_id"], prd, spec.get("itm_id", "ALL"), obj, part[0], part[-1])
            rows.extend(got)
        except KosisError as e:
            if is_too_many(e) and len(part) > 1:
                half = len(part) // 2
                stack.append(part[half:]); stack.append(part[:half])
                continue
            raise
        time.sleep(SLEEP_SEC)
    return rows


def latest_period_string(prd):
    """오늘 기준 요청 가능한 마지막 시점 (미발표면 빈 응답으로 넘어감)."""
    t = date.today()
    if prd == "M":
        return t.strftime("%Y%m")
    if prd == "Q":
        return f"{t.year}{(t.month - 1) // 3 + 1:02d}"
    return str(t.year)

print("API 함수 준비 완료")

## 4. DB 테이블과 적재 함수

In [ ]:
from sqlalchemy import text

T_INFO, T_DATA, T_VIN = "kr_kosis_series_info", "kr_kosis_data", "kr_kosis_data_vintage"
_OPTS = "ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci"
DDL = [
f"""
CREATE TABLE IF NOT EXISTS {T_INFO} (
    series_id          VARCHAR(50)  NOT NULL COMMENT 'kosis_ + 해시 16자리',
    series_hash        CHAR(64)     NOT NULL COMMENT '기관+통계표+항목+분류코드 SHA-256',
    table_alias        VARCHAR(50)  NULL     COMMENT '등록표 alias',
    org_id             VARCHAR(10)  NOT NULL,
    tbl_id             VARCHAR(50)  NOT NULL,
    tbl_name           VARCHAR(255) NULL,
    itm_id             VARCHAR(50)  NULL,
    itm_name           VARCHAR(255) NULL,
    c1_code VARCHAR(50) NULL, c1_name VARCHAR(255) NULL,
    c2_code VARCHAR(50) NULL, c2_name VARCHAR(255) NULL,
    c3_code VARCHAR(50) NULL, c3_name VARCHAR(255) NULL,
    dims_json          TEXT         NULL     COMMENT '분류 C1~C8 코드/이름/분류명 전체',
    dims_label         VARCHAR(500) NULL     COMMENT '분류 이름을 > 로 이은 표시용 이름',
    freq               CHAR(1)      NOT NULL COMMENT 'M,Q,Y',
    unit               VARCHAR(100) NULL,
    source_last_change DATE         NULL     COMMENT 'KOSIS 통계표 최종 갱신일 (LST_CHN_DE)',
    is_active          TINYINT      NOT NULL DEFAULT 1,
    first_collected_at DATETIME     NOT NULL,
    last_collected_at  DATETIME     NOT NULL,
    created_at         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id),
    UNIQUE KEY uk_series_hash (series_hash),
    KEY idx_table (org_id, tbl_id),
    KEY idx_alias (table_alias)
) {_OPTS} COMMENT='KOSIS 통계 시리즈 설명표 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_DATA} (
    series_id          VARCHAR(50) NOT NULL,
    period_end         DATE        NOT NULL COMMENT '기간 말일',
    freq               CHAR(1)     NOT NULL,
    value              DOUBLE      NULL,
    first_collected_at DATETIME    NOT NULL COMMENT '최초 관측 시각(KST)',
    last_collected_at  DATETIME    NOT NULL COMMENT '최근 관측 시각(KST)',
    created_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at         TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id, period_end),
    KEY idx_period_end (period_end)
) {_OPTS} COMMENT='KOSIS 통계 값 (STD-1)'
""",
f"""
CREATE TABLE IF NOT EXISTS {T_VIN} (
    id           BIGINT      NOT NULL AUTO_INCREMENT,
    series_id    VARCHAR(50) NOT NULL,
    period_end   DATE        NOT NULL,
    value        DOUBLE      NULL,
    change_type  VARCHAR(10) NOT NULL COMMENT 'new | revised',
    collected_at DATETIME    NOT NULL COMMENT '이 값을 관측한 시각(KST)',
    created_at   TIMESTAMP   NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id),
    KEY idx_key_time (series_id, period_end, collected_at)
) {_OPTS} COMMENT='KOSIS 통계 관측 이력 (STD-1 vintage)'
""",
]
if SAVE_DB:
    with engine.begin() as cx:
        for ddl in DDL:
            cx.execute(text(ddl))
    print("테이블 준비 완료:", T_INFO, T_DATA, T_VIN)


def now_kst():
    return datetime.now(KST).replace(tzinfo=None, microsecond=0)


def to_period_end(prd_de, prd):
    """KOSIS 시점 → 기간 말일. 월 YYYYMM, 분기 YYYYQQ(01~04), 반기 YYYYHH, 연 YYYY, 일 YYYYMMDD."""
    s = str(prd_de).strip()
    y = int(s[:4])
    if prd == "M":
        return (pd.Timestamp(year=y, month=int(s[4:6]), day=1) + pd.offsets.MonthEnd(0)).date()
    if prd == "Q":
        q = int(s[4:])
        return (pd.Timestamp(year=y, month=3 * q, day=1) + pd.offsets.MonthEnd(0)).date()
    if prd == "H":
        return date(y, 6, 30) if int(s[4:]) == 1 else date(y, 12, 31)
    if prd == "Y":
        return date(y, 12, 31)
    return datetime.strptime(s[:8], "%Y%m%d").date()


def period_string(d, prd):
    d = pd.Timestamp(d)
    if prd == "M":
        return d.strftime("%Y%m")
    if prd == "Q":
        return f"{d.year}{d.quarter:02d}"
    return str(d.year)


def _num(v):
    s = str(v if v is not None else "").replace(",", "").strip()
    try:
        x = float(s)
        return x if np.isfinite(x) else None
    except ValueError:
        return None                 # '-', 'x', '' 등 비공개/결측


def _same(a, b):
    if a is None or b is None:
        return a is None and b is None
    return abs(float(a) - float(b)) <= 1e-9 * max(1.0, abs(float(a)))


def rows_to_records(rows, spec, prd):
    """KOSIS 응답 행 → (설명표 dict, 값 list)"""
    info, data = {}, []
    for r in rows:
        dims = {}
        for lv in range(1, 9):
            c = str(r.get(f"C{lv}") or "").strip()
            if c:
                dims[f"C{lv}"] = {"code": c, "name": str(r.get(f"C{lv}_NM") or "").strip(),
                                  "obj": str(r.get(f"C{lv}_OBJ_NM") or "").strip()}
        itm = str(r.get("ITM_ID") or "").strip()
        key = json.dumps([spec["org_id"], spec["tbl_id"], itm,
                          [(k, v["code"]) for k, v in sorted(dims.items())]], ensure_ascii=False)
        h = hashlib.sha256(key.encode("utf-8")).hexdigest()
        sid = "kosis_" + h[:16]
        if sid not in info:
            lcd = str(r.get("LST_CHN_DE") or "").strip()
            info[sid] = {
                "series_id": sid, "series_hash": h, "table_alias": spec["alias"],
                "org_id": spec["org_id"], "tbl_id": spec["tbl_id"],
                "tbl_name": (str(r.get("TBL_NM") or "").strip() or None),
                "itm_id": itm or None, "itm_name": (str(r.get("ITM_NM") or "").strip() or None),
                "c1_code": dims.get("C1", {}).get("code"), "c1_name": dims.get("C1", {}).get("name"),
                "c2_code": dims.get("C2", {}).get("code"), "c2_name": dims.get("C2", {}).get("name"),
                "c3_code": dims.get("C3", {}).get("code"), "c3_name": dims.get("C3", {}).get("name"),
                "dims_json": json.dumps(dims, ensure_ascii=False),
                "dims_label": " > ".join(v["name"] for _, v in sorted(dims.items()))[:500] or None,
                "freq": prd, "unit": (str(r.get("UNIT_NM") or "").strip() or None),
                "source_last_change": (pd.to_datetime(lcd[:8], format="%Y%m%d", errors="coerce").date()
                                       if len(lcd) >= 8 else None),
            }
        val = _num(r.get("DT"))
        if val is None:
            continue
        data.append({"series_id": sid, "period_end": to_period_end(r.get("PRD_DE"), prd), "value": val})
    return info, data


def save_kosis(info, data):
    """반환: (시리즈 수, 저장 건수, 신규, 수정)"""
    ts = now_kst()
    infos = [dict(v, ts=ts) for v in info.values()]
    dd = {}
    for d in data:
        dd[(d["series_id"], d["period_end"])] = d          # 같은 키 중복이면 마지막 값
    recs = [dict(v, ts=ts) for v in dd.values()]
    if not recs:
        return len(infos), 0, 0, 0
    cols = ["series_id", "series_hash", "table_alias", "org_id", "tbl_id", "tbl_name", "itm_id", "itm_name",
            "c1_code", "c1_name", "c2_code", "c2_name", "c3_code", "c3_name", "dims_json", "dims_label",
            "freq", "unit", "source_last_change"]
    sql_info = f"""INSERT INTO {T_INFO} ({", ".join(cols)}, first_collected_at, last_collected_at)
        VALUES ({", ".join(":" + c for c in cols)}, :ts, :ts)
        ON DUPLICATE KEY UPDATE table_alias = VALUES(table_alias), tbl_name = VALUES(tbl_name),
            itm_name = VALUES(itm_name), c1_name = VALUES(c1_name), c2_name = VALUES(c2_name),
            c3_name = VALUES(c3_name), dims_json = VALUES(dims_json), dims_label = VALUES(dims_label),
            unit = VALUES(unit), source_last_change = VALUES(source_last_change),
            last_collected_at = VALUES(last_collected_at)"""
    sql_data = f"""INSERT INTO {T_DATA} (series_id, period_end, freq, value, first_collected_at, last_collected_at)
        VALUES (:series_id, :period_end, :freq, :value, :ts, :ts)
        ON DUPLICATE KEY UPDATE value = VALUES(value),
            first_collected_at = LEAST(first_collected_at, VALUES(first_collected_at)),
            last_collected_at  = GREATEST(last_collected_at, VALUES(last_collected_at))"""
    sql_vin = f"""INSERT INTO {T_VIN} (series_id, period_end, value, change_type, collected_at)
        VALUES (:series_id, :period_end, :value, :change_type, :ts)"""
    freq_of = {v["series_id"]: v["freq"] for v in infos}
    for r in recs:
        r["freq"] = freq_of[r["series_id"]]
    sids = sorted(freq_of)
    with engine.begin() as cx:
        existing = {}
        for i in range(0, len(sids), 500):
            part = sids[i:i + 500]
            q = text(f"SELECT series_id, period_end, value FROM {T_DATA} WHERE series_id IN ("
                     + ", ".join(f":s{j}" for j in range(len(part))) + ")")
            for s, p, v in cx.execute(q, {f"s{j}": x for j, x in enumerate(part)}):
                existing[(s, p)] = v
        vin = []
        for r in recs:
            k = (r["series_id"], r["period_end"])
            if k not in existing:
                vin.append(dict(r, change_type="new"))
            elif not _same(existing[k], r["value"]):
                vin.append(dict(r, change_type="revised"))
        for i in range(0, len(infos), 1000):
            cx.execute(text(sql_info), infos[i:i + 1000])
        for i in range(0, len(recs), 2000):
            cx.execute(text(sql_data), recs[i:i + 2000])
        for i in range(0, len(vin), 2000):
            cx.execute(text(sql_vin), vin[i:i + 2000])
    n_new = sum(1 for v in vin if v["change_type"] == "new")
    return len(infos), len(recs), n_new, len(vin) - n_new


def db_last_period(spec):
    with engine.connect() as cx:
        return cx.execute(text(f"SELECT MAX(d.period_end) FROM {T_DATA} d JOIN {T_INFO} i "
                               f"ON d.series_id = i.series_id WHERE i.org_id = :o AND i.tbl_id = :t"),
                          {"o": spec["org_id"], "t": spec["tbl_id"]}).scalar()


def load_kosis_long(alias=None, tbl_id=None, start=None):
    q = (f"SELECT i.table_alias, i.tbl_name, i.itm_name, i.dims_label, i.unit, i.series_id, "
         f"d.period_end, d.value FROM {T_DATA} d JOIN {T_INFO} i ON d.series_id = i.series_id WHERE 1 = 1")
    p = {}
    if alias:
        q += " AND i.table_alias = :a"; p["a"] = alias
    if tbl_id:
        q += " AND i.tbl_id = :t"; p["t"] = tbl_id
    if start:
        q += " AND d.period_end >= :s"; p["s"] = start
    with engine.connect() as cx:
        out = pd.read_sql(text(q), cx, params=p)
    out["period_end"] = pd.to_datetime(out["period_end"])
    out["label"] = out["dims_label"].fillna("") + " | " + out["itm_name"].fillna("")
    return out

print("DB 함수 준비 완료")

## 5. 통계표 확인 (최근 1개 시점만 조회, DB 저장 없음)
통계표 이름, 주기, 분류 단계, 항목 수를 보여줍니다. **이름이 원하는 통계가 맞는지 확인**하고, 아니면 1번 셀에서 `enabled = False`로 바꾸세요.

In [ ]:
STRUCT = {}
for spec in KOSIS_TABLES:
    if not spec.get("enabled", True):
        continue
    try:
        prd, obj, rows = detect_structure(spec)
        STRUCT[spec["alias"]] = (prd, obj)
        df0 = pd.DataFrame(rows)
        print("=" * 70)
        print(f"[{spec['alias']}] {spec['org_id']}/{spec['tbl_id']}")
        print("  통계표명 :", df0.get("TBL_NM", pd.Series([""])).iloc[0])
        print("  주기     :", prd, "| 분류 단계:", len(obj), "| 최근 시점:", df0["PRD_DE"].max())
        print("  항목     :", df0[["ITM_ID", "ITM_NM"]].drop_duplicates().head(8).values.tolist())
        for lv in range(1, 9):
            if f"C{lv}" in df0:
                names = df0[f"C{lv}_NM"].drop_duplicates().tolist()
                print(f"  분류 C{lv}  : {len(names)}개  예) {names[:6]}")
        print("  단위     :", list(df0.get("UNIT_NM", pd.Series([""])).dropna().unique()[:3]))
    except Exception as e:
        print(f"[{spec['alias']}] 확인 실패: {_mask(e)}")

## 6. 수집 및 적재
- `MODE = "auto"`: DB에 있으면 마지막 시점 기준 최근 `REFRESH_PERIODS`기간만, 없으면 `start`부터 전체

In [3]:
summary = []
for spec in KOSIS_TABLES:
    if not spec.get("enabled", True):
        continue
    alias = spec["alias"]
    try:
        prd, obj = STRUCT.get(alias) or detect_structure(spec)[:2]
        start = spec["start"]
        if SAVE_DB and MODE == "auto":
            last = db_last_period(spec)
            if last is not None:
                start = max(start, _shift_period(period_string(last, prd), prd, -(REFRESH_PERIODS - 1)))
        end = latest_period_string(prd)
        print(f"[{alias}] {start} ~ {end} 수집 중 ...")
        rows = fetch_range(spec, prd, obj, start, end)
        info, data = rows_to_records(rows, spec, prd)
        if SAVE_CSV and rows:
            os.makedirs(OUT_DIR, exist_ok=True)
            pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, f"kosis_{alias}_{datetime.now():%Y%m%d}.csv"),
                                      index=False, encoding="utf-8-sig")
        if SAVE_DB:
            n_s, n, n_new, n_rev = save_kosis(info, data)
        else:
            n_s, n, n_new, n_rev = len(info), len(data), 0, 0
        first = min((d["period_end"] for d in data), default=None)
        lastp = max((d["period_end"] for d in data), default=None)
        print(f"   응답 {len(rows):,}행 → 시리즈 {n_s}개, 값 {n:,}건 (신규 {n_new:,}, 수정 {n_rev:,}) {first} ~ {lastp}")
        summary.append((alias, "OK", n_s, n, n_new, n_rev, first, lastp))
    except Exception as e:
        print(f"[{alias}] 실패: {_mask(e)}")
        summary.append((alias, f"FAIL: {_mask(e)[:80]}", 0, 0, 0, 0, None, None))

pd.DataFrame(summary, columns=["alias", "status", "n_series", "n_values", "new", "revised", "first", "last"])

NameError: name '_mask' is not defined

## 7. 적재 결과 검증

In [ ]:
if SAVE_DB:
    with engine.connect() as cx:
        chk = pd.read_sql(text(f"""
            SELECT i.table_alias, i.tbl_id, MAX(i.tbl_name) AS tbl_name, i.freq,
                   COUNT(DISTINCT i.series_id) AS n_series, COUNT(d.period_end) AS n_values,
                   MIN(d.period_end) AS first_period, MAX(d.period_end) AS last_period,
                   MAX(i.source_last_change) AS kosis_updated
            FROM {T_INFO} i LEFT JOIN {T_DATA} d ON d.series_id = i.series_id
            GROUP BY i.table_alias, i.tbl_id, i.freq"""), cx)
    pd.set_option("display.width", 200)
    print(chk.to_string(index=False))

## 8. 분석 예시: 소매판매액지수 (DB에서 읽기)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import platform
from matplotlib import font_manager

for name in ({"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"]}.get(platform.system(), []) + ["NanumGothic"]):
    if name in {f.name for f in font_manager.fontManager.ttflist}:
        matplotlib.rcParams["font.family"] = name
        break
matplotlib.rcParams["axes.unicode_minus"] = False

if SAVE_DB:
    rs = load_kosis_long(alias="retail_sales_index", start="2015-01-01")
    if rs.empty:
        print("소매판매액지수 데이터가 없습니다.")
    else:
        wide = rs.pivot_table(index="period_end", columns="label", values="value", aggfunc="last")
        print("시리즈 수:", wide.shape[1], "| 기간:", wide.index.min().date(), "~", wide.index.max().date())
        print("\n시리즈 이름 예시:")
        for c in list(wide.columns)[:10]:
            print("  ", c)
        # 첫 분류(보통 '총지수') 계열만 그림
        cols = [c for c in wide.columns if "총지수" in c][:4] or list(wide.columns[:4])
        ax = wide[cols].plot(figsize=(12, 5), title="소매판매액지수 (KOSIS DT_1K41012)")
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()